# Lecture 10 — Higher-order functions and decorators

A Python function is an object, so a function can take functions as arguments and return new ones.
Wrappers built this way count, time and cache calls without editing the functions they wrap.

Notes: [Lecture 10](https://teaching.borisbolliet.io/c1-2026/materials/class10/notes.pdf) ·
code: [`attractor` at `lecture-10`](https://github.com/borisbolliet/attractor/tree/lecture-10/)

The notebook runs at the root of the `attractor` clone, at the tag `lecture-10`, with the kernel
*Python (attractor 3.13)* that `setup_env.sh` installs:

```bash
cd ~/GitHub/attractor
git fetch --tags --force
git switch --detach lecture-10
source setup_env.sh
```

In [1]:
import functools, inspect, math, os, subprocess, sys, tempfile, time
from pathlib import Path
import numpy as np

TAG = "lecture-10"
if not Path("tools.py").exists():   # outside a clone (Colab, a download): fetch the project
    root = Path(tempfile.mkdtemp()) / "attractor"
    subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "-q", "--depth", "1", "--branch", TAG,
                    "https://github.com/borisbolliet/attractor.git", str(root)], check=True)
    os.chdir(root)
sys.path.insert(0, os.getcwd())
subprocess.run(["git", "describe", "--tags"], capture_output=True, text=True).stdout.strip()

'lecture-10'

## 1 Functions that take and return functions

### Higher-order functions

A **higher-order function** takes a callable as an argument, returns one, or does both; one of the
two is enough. `make_adder(n)` of the previous lecture returns a new function, `add`. A higher-order
function that builds and returns a new function in this way is a **factory**.

`sorted` is a higher-order function: its argument `key` is a function, which `sorted` calls once per
element and orders by the results.

In [2]:
sorted(["pear", "fig", "apple"])             # alphabetical

['apple', 'fig', 'pear']

In [3]:
sorted(["pear", "fig", "apple"], key=len)    # by len of each word: 3, 4, 5

['fig', 'pear', 'apple']

`functools.partial` takes a function and returns a new **callable**, an object called like a function,
with some arguments fixed. `partial` is the factory, and `from_binary` is what it returns: a partial
object, callable but not a function.

In [4]:
from_binary = functools.partial(int, base=2)   # takes int, returns a new callable
type(from_binary), callable(from_binary)

(functools.partial, True)

In [5]:
from_binary("101")                              # int("101", base=2)

5

In the project, `make_lotka_volterra` of [`models.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/models.py) takes four rates and returns a new function,
`rhs`, so it is a factory. `integrate` of [`integrators.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/integrators.py) takes two callables, `step` and `rhs`,
and returns arrays, so it is higher-order but not a factory.

In [6]:
from integrators import integrate, rk4_step
from models import CHEMOSTAT, make_lotka_volterra

rhs = make_lotka_volterra(**CHEMOSTAT)      # the factory returns a function
rhs, rhs(0.0, np.array([1.5, 8.0]))

(<function models.make_lotka_volterra.<locals>.rhs(t, y)>, array([0.24, 3.08]))

In [7]:
print(inspect.getsource(integrate))          # step is called, rhs is passed on to it

def integrate(step, rhs, y0, t0, t1, h):
    """Integrate from t0 to t1 with a fixed step; returns (t, y) with y of shape (n+1, len(y0))."""
    n = round((t1 - t0) / h)
    y = np.array(y0, dtype=float)
    ts, ys = [t0], [y]
    for i in range(n):
        y = step(rhs, t0 + i * h, y, h)
        ts.append(t0 + (i + 1) * h)
        ys.append(y)
    return np.array(ts), np.array(ys)



In [8]:
t, y = integrate(rk4_step, rhs, np.array([1.5, 8.0]), 0.0, 40.0, 0.05)   # two functions in, arrays out
y.shape

(801, 2)

### Wrappers

A **wrapper** accepts any arguments, forwards them to a function stored when the wrapper was made,
and returns that function's result, with work of its own before or after the call.
It is a closure: the stored function is a free variable of the wrapper, kept in a cell.

In [9]:
def counted(f):
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return f(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

hyp = counted(math.hypot)
hyp(3, 4), hyp(5, 12), hyp.calls

(5.0, 13.0, 2)

`counted` is higher-order: it takes `f` and returns `wrapper`. `wrapper` uses the name `f`, which is not
its parameter but the parameter of `counted`: a **free variable**. When `counted` returns, its local names
would disappear with its call, so Python keeps `f` in a **cell** attached to `wrapper`. The function with
its cells is a closure. `wrapper` is a free variable too, since the body reads `wrapper.calls`.

In [10]:
hyp.__code__.co_freevars, hyp.__closure__[0].cell_contents

(('f', 'wrapper'), <function math.hypot>)

Each call of `counted` makes a new wrapper with its own cells, around one shared code object:

In [11]:
hyp2 = counted(math.hypot)
hyp2 is hyp, hyp2.__code__ is hyp.__code__, hyp2.calls

(False, True, 0)

A wrapper must call its parameter. In [`experiments/wrong_name.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/experiments/wrong_name.py) the body calls the global `f`
instead of the parameter `g`, so the wrapped `square` doubles, and the wrapper has no cell for it:

In [12]:
%run -m experiments.wrong_name

6 6
('wrapper',)


### Forwarding of arguments

`*args` collects the positional arguments of a call into a tuple and `**kwargs` the keyword arguments
into a dictionary. `f(*args, **kwargs)` unpacks them again, so one wrapper serves functions of any
**signature**, the list of parameters a function accepts:

In [13]:
def add(x, y):
    return x + y

args, kwargs = (3, 4), {"x": 3, "y": 4}
add(*args), add(**kwargs)          # add(3, 4) and add(x=3, y=4)

(7, 7)

In [14]:
add(args)                          # one argument: the tuple

TypeError: add() missing 1 required positional argument: 'y'

A wrapper with named parameters fits one signature only:

In [ ]:
def counted_badly(f):
    def wrapper(x):
        wrapper.calls += 1
        return f(x)
    wrapper.calls = 0
    return wrapper

counted_badly(abs)(-2)

In [ ]:
counted_badly(math.hypot)(3, 4)

### Metadata of a wrapped function

A wrapper is a new function object named `wrapper`, without the docstring of the function it wraps,
and `inspect.signature` reports `(*args, **kwargs)`. `@functools.wraps(f)` above the inner `def`
copies the name, qualified name, docstring and annotations of `f`, and sets `__wrapped__`,
which `inspect.signature` follows.

In [ ]:
def area(w, h):
    """Area of a w by h rectangle."""
    return w * h

def bare(f):
    def wrapper(*args, **kwargs):
        return f(*args, **kwargs)
    return wrapper

def neat(f):
    @functools.wraps(f)
    def wrapper(*args, **kwargs):
        return f(*args, **kwargs)
    return wrapper

b, n = bare(area), neat(area)
b.__name__, b.__doc__, inspect.signature(b)

In [ ]:
n.__name__, n.__doc__, inspect.signature(n), n.__wrapped__ is area

The copy is made once, when the decorator is applied, and the code object is not copied:
`n.__qualname__` is that of `area`, while the code that runs is still the wrapper's.

In [ ]:
n.__qualname__, n.__code__.co_qualname

## 2 Decorators

### The decorator syntax

A **decorator** is a callable applied to a function right after the `def` statement has created it.
Its return value is bound to the function's name: `@deco` above `def f` is `f = deco(f)`.
The line below appears when the `def` runs, before any call.

In [ ]:
def loud(f):
    print("decorating", f.__name__)
    return f

@loud
def area(w, h):
    return w * h

In [ ]:
area = loud(area); area(2, 3)

The counter of [`tools.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/tools.py) on the chemostat right-hand side, in the long spelling
`rhs = counted(rhs)`: 800 RK4 steps make 4 × 800 = 3200 calls.

In [ ]:
from tools import counted
rhs = counted(make_lotka_volterra(**CHEMOSTAT))
t, y = integrate(rk4_step, rhs, np.array([1.5, 8.0]), 0.0, 40.0, 0.05)
y.shape, rhs.calls

In [ ]:
rhs.reset(); rhs(0.0, np.array([1.0, 10.0])), rhs.calls

### Decorator factories

A **decorator factory** is a function whose return value is a decorator. In `@factory(arg)` the call
runs first and returns the decorator, which is then applied. `functools.lru_cache(maxsize=32)` is one,
and the bare `@functools.lru_cache` keeps the default of 128 entries.

In [ ]:
@functools.lru_cache(maxsize=32)
def square(n):
    return n * n

@functools.lru_cache
def cube(n):
    return n ** 3

square.cache_info().maxsize, cube.cache_info().maxsize

A factory written by hand nests three functions: the factory returns the decorator, which returns the
wrapper. The factory's argument is a free variable of the wrapper:

In [ ]:
def announce(message):                 # factory
    def decorator(f):                  # decorator
        def wrapper(*args, **kwargs):  # wrapper
            print(message)
            return f(*args, **kwargs)
        return wrapper
    return decorator

@announce("Bonjour!")
def double(x):
    return 2 * x

double(3), double.__code__.co_freevars

[`tools.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/tools.py) holds a dictionary `REGISTRY` and a factory `register(name)`, whose decorator stores
the decorated function under `name` and returns it unchanged:

In [ ]:
import tools
print(inspect.getsource(tools.register))

### Import time and call time

The top-level code of a module runs once, at its first import, and the module is then kept in
`sys.modules`. A `def` and its decorator are top-level code, so a decorator's body runs at
**import time**, once per decorated function, and its wrapper's body at **call time**, once per call.
[`experiments/import_time.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/experiments/import_time.py) prints in both; the second import prints nothing:

In [ ]:
!{sys.executable} -c "import experiments.import_time as m; m.area(2, 3); m.area(4, 5); import experiments.import_time"

A **registry** is a dictionary from names to functions, filled by a decorator as each function is
defined. Importing [`models.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/models.py) fills `REGISTRY`, and a right-hand side is then built by name:

In [ ]:
from tools import REGISTRY, make_model
import models
sorted(REGISTRY), REGISTRY is models.REGISTRY

In [ ]:
make_model("lotka_volterra", **models.CHEMOSTAT).__qualname__

### Stacked decorators

**Stacked** decorators apply from the bottom up: `@A` over `@B` over `def g` is `g = A(B(g))`.
[`experiments/stack_order.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/experiments/stack_order.py) prints which decorator is applied first:

In [ ]:
%run -m experiments.stack_order

## 3 Counting and timing

### Counting calls

A count of calls depends on the algorithm and its input only. Under the counter, a recursive function
counts its inner calls too, since the recursion goes through the name that the decoration rebound:

In [ ]:
@counted
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

fib(10), fib.calls

In [ ]:
fib.reset(); fib(20), fib.calls

[`experiments/cost.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/experiments/cost.py) counts the right-hand-side calls of Euler's method and RK4 for five step
sizes, against the error at t = 40 days:

In [ ]:
%run -m experiments.cost

### Timers, overhead and profiles

`time.perf_counter()` reads the clock for intervals. It never goes back and its zero means nothing,
so only a difference of two readings counts. A timing wrapper, `timed` of [`tools.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/tools.py), takes the
two readings around the forwarded call. Its own work, the **overhead**, is about 100 ns per call.

In [ ]:
start = time.perf_counter()
total = sum(range(10**6))
elapsed = time.perf_counter() - start
total, elapsed

A **profiler** records every call of a run with its time. `cProfile` reports for each function
`ncalls`, `tottime`, the time in its own lines, and `cumtime`, the time including the functions it calls.
[`experiments/timing.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/experiments/timing.py) profiles one run of `integrate` with the right-hand side under `counted`:

In [ ]:
%run -m experiments.timing

A profiler answers which functions take the time of a run, and its bookkeeping slows every call it
records. A benchmark answers how long one operation takes, with repeated runs of the operation alone.

## 4 Caching

### Memoisation

**Memoisation** stores the result of a call under its arguments and returns it when the same arguments
come again. It is correct when the result depends on the arguments alone. `functools.lru_cache` keeps a
**cache**, a dictionary keyed by the arguments, so the arguments must be **hashable**, with a hash that
never changes. A call answered from the cache is a **hit**, one computed and stored a **miss**.

In [ ]:
@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

fib(80), fib.cache_info()

A cache pays off when a call is slow:

In [ ]:
@functools.lru_cache(maxsize=32)
def slow_square(n):
    time.sleep(0.5)
    return n * n

for _ in range(2):
    start = time.perf_counter()
    slow_square(30)
    print(f"{time.perf_counter() - start:.3f} s")
slow_square.cache_info()

A key is found by its hash and then by `==`, not by identity. Two tuples built at run time are two
objects with equal hashes, and each finds the entry stored under the other. A list has no hash: its
contents can change.

In [ ]:
a, b = tuple([1, 2]), tuple([1, 2])
a == b, hash(a) == hash(b), a is b

In [ ]:
fib([80])

### Cached objects and aliasing

A cache returns the stored object itself, not a copy. A caller that writes into a cached result
changes what every later caller receives, and nothing raises.

In [ ]:
@functools.lru_cache(maxsize=32)
def make_list(n):
    return [0] * n

a = make_list(3)
b = make_list(3)
a[0] = 99
a, b, a is b, make_list(3)       # the body says [0] * n, the cache returns the changed list

[`experiments/fib_cache.py`](https://github.com/borisbolliet/attractor/blob/lecture-10/experiments/fib_cache.py) caches a loader of the chemostat runs by path. Two calls return one
array, and a write through the first result changes the second:

In [ ]:
%run -m experiments.fib_cache

The exercises, with their solutions, are at the end of the notes; their scripts are in
[`exercises/lecture10/`](https://github.com/borisbolliet/attractor/tree/lecture-10/exercises/lecture10).